In [ ]:
import sys, os
from pathlib import Path

import requests
import ast
import logging
import openai
import torch
from tqdm import tqdm

from vllm import LLM, SamplingParams
import torch.multiprocessing as mp
from transformers import AutoTokenizer, AutoModel,AutoConfig

mp.set_start_method('spawn', force=True)

logging.basicConfig(level=logging.INFO)
logging.getLogger("vllm").setLevel(logging.WARNING)
sys.path.insert(0, str(Path(".").resolve().parent))


import src.preprocessing.utils.biomarker_CiViC_filter as biomarker_CiViC_filter   
import src.preprocessing.utils.parsing as parsing_ClinicalTrials
import src.preprocessing.utils.prompt_call as prompt_call_ClinicalTrials
import src.preprocessing.utils.embeddings as embeddings_ClinicalTrials



/users/jcc2340/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
CUDA available: True


In [40]:
import importlib
importlib.reload(embeddings_ClinicalTrials)

CUDA available: True


<module 'src.preprocessing.utils.embeddings' from '/groups/gg2845_gp/jcc2340/projects/precision_oncology/src/preprocessing/utils/embeddings.py'>

In [33]:
clinical_trials_path="../data/synthetic/trials/TREC_2023/ClinicalTrials.2023-05-08.trials5/NCT0585xxxx/"
cancer_filter=True
filter_civic=False
model="Qwen/Qwen1.5-14B-Chat"
embedding_model="ncbi/MedCPT-Query-Encoder"
save_dir="../database/chromadb/"

In [15]:
if cancer_filter:
     filter_words=["cancer", "tumor", "neoplasm"]
     
trials = parsing_ClinicalTrials.process_clinical_trials(clinical_trials_path,
                                                            word_filter=filter_words)


In [19]:
dnf_prompt_dir = Path("../src/prompts/trialQuestionsPrompt.py")
PROMPT_DNF_TEMPLATE = prompt_call_ClinicalTrials.load_DNF_prompt(dnf_prompt_dir)
config= {
          	'model_name': model,
          	'temperature': 0.0,
               'max_tokens': 1024,
               'max_context': 0,

          }
def_conf=AutoConfig.from_pretrained(config['model_name'])
config['max_context'] = getattr(def_conf, "max_position_embeddings", None)


In [21]:
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  
llm = LLM(
        model=config['model_name'],
        gpu_memory_utilization=0.88,
        dtype='bfloat16',
        max_model_len=13472
        )
print(f"LLM model '{model}' initialized successfully.")

(EngineCore pid=3734195) INFO 07-06 20:03:57 [core.py:113] Initializing a V1 LLM engine (v0.23.0) with config: model='Qwen/Qwen1.5-14B-Chat', speculative_config=None, tokenizer='Qwen/Qwen1.5-14B-Chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=13472, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:00<00:02,  2.41it/s]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:00<00:02,  2.04it/s]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:01<00:02,  1.99it/s]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:01<00:02,  1.98it/s]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:02<00:01,  1.97it/s]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:03<00:01,  1.97it/s]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:03<00:00,  2.09it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:03<00:00,  2.50it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:03<00:00,  2.19it/s]
(EngineCore pid=3734195) 


(EngineCore pid=3734195) INFO 07-06 20:04:04 [default_loader.py:397] Loading weights took 3.70 seconds
(EngineCore pid=3734195) INFO 07-06 20:04:05 [gpu_model_runner.py:5187] Model loading took 26.43 GiB memory and 4.805557 seconds
(EngineCore pid=3734195) INFO 07-06 20:04:07 [backends.py:1089] Using cache directory: /users/jcc2340/.cache/vllm/torch_compile_cache/58602afa2a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3734195) INFO 07-06 20:04:07 [backends.py:1148] Dynamo bytecode transform time: 2.46 s
(EngineCore pid=3734195) INFO 07-06 20:04:09 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.285 s
(EngineCore pid=3734195) INFO 07-06 20:04:09 [decorators.py:311] Directly load AOT compilation from path /users/jcc2340/.cache/vllm/torch_compile_cache/torch_aot_compile/235ba7764416c2efa1f16c94d5d5001b38c424a1f970351bd702c803e1a80eea/rank_0_0/model
(EngineCore pid=3734195) INFO 07-06 20:04:09 [monitor.py:53] torch.comp

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.60it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.68it/s]


(EngineCore pid=3734195) INFO 07-06 20:04:19 [gpu_model_runner.py:6585] Graph capturing finished in 7 secs, took 0.54 GiB
(EngineCore pid=3734195) INFO 07-06 20:04:19 [gpu_worker.py:639] CUDA graph pool memory: 0.54 GiB (actual), 0.62 GiB (estimated), difference: 0.08 GiB (15.1%).
(EngineCore pid=3734195) INFO 07-06 20:04:19 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=3734195) INFO 07-06 20:04:19 [core.py:306] init engine (profile, create kv cache, warmup model) took 14.65 s (compilation: 3.97 s)
(EngineCore pid=3734195) INFO 07-06 20:04:20 [vllm.py:999] Asynchronous scheduling is enabled.
(EngineCore pid=3734195) INFO 07-06 20:04:20 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
LLM model 'Qwen/Qwen1.5-14B-Chat' initialized successfully.


In [38]:
# I think we can do it till here and then just generate the pickle file and then open it when we do the matches

In [ ]:
for trial in tqdm(trials, desc="Generating DNF", unit="trial", dynamic_ncols=True):
                         prompt_call_ClinicalTrials.generate_DNF(trial, PROMPT_DNF_TEMPLATE, config, type_run='hpc', llm=llm)

embeddings_ClinicalTrials.save_questions_to_pickle(trials,Path(save_dir))

Generating DNF:   0%|          | 0/17 [00:00<?, ?trial/s]

<|im_start|>system
You are an expert in clinical trial eligibility criteria. Output only valid JSON, no extra text or explanations.<|im_end|>
<|im_start|>user
You are an expert clinical trial data extractor. Convert the given INCLUSION and EXCLUSION CRITERIA into a set of clear, grammatically correct YES/NO questions, then build a logical (DNF) expression over those questions to determine trial eligibility.

INPUT FORMAT:
Two strings — "Inclusion Criteria" and "Exclusion Criteria" (the latter's header starts with "!"). Each is formatted as "Header: criteria1 criteria2 ...", with individual criteria separated by newlines, bullets, hyphens, or numbers.

QUESTION RULES:
- One question per single criterion. If a line contains multiple criteria, split it into separate questions (e.g. "18-65 years old and ECOG 0-1" > "Is the patient between 18 and 65 years old?" + "Does the patient have an ECOG performance status of 0 or 1?").
- Questions must be answerable independently, and phrased only in

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00,  5.29it/s]


(EngineCore pid=3734195) WARNING 07-06 20:04:39 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Generating DNF:   6%|▌         | 1/17 [00:21<05:41, 21.32s/trial]

...

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Volunteers to participate and sign informed consent?",', '        "Q2": "Age >= 18 years old?",', '        "Q3": "Female or male?",', '        "Q4": "ECOG score 0-1?",', '        "Q5": "Expected survival ≥ 12 weeks?",', '        "Q6": "Met criteria for ER positive and HER2 negative female breast cancer?",', '        "Q7": "At least one extracranial measurable lesion per RECIST v1.1?",', '        "Q8": "Normal function of important organs?",', '        "Q9": "Nitrosourea or mitomycin C interval ≥ 6 weeks?",', '        "Q10": "Cytotoxic drug interval ≥ 4 weeks?",', '        "Q11": "Radiotherapy interval ≥ 2 weeks?",', '        "Q12": "Adverse events severity ≤ 1 (excluding hair loss)?",', '        "Q13": "Female subjects using contraception?",', '        "Q14": "Male subjects using contraception?",', '        "Q15": "Female subjects non-lactating and HCG negative?",', '        "Q16": "Volunteer to follow study procedures?",', '      

Generating DNF:  94%|█████████▍| 16/17 [03:42<00:14, 14.17s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Does the patient have written informed consent according to ICH/GCP regulations before registration?",', '        "Q2": "Is the patient at least 18 years old?",', '        "Q3": "Does the patient have a histology of adenocarcinoma of the prostate?",', '        "Q4": "Is the patient receiving ADT alone or ADT+NHT (abiraterone, enzalutamide, apalutamide, or darolutamide)?",', '        "Q5": "Has the patient experienced two consecutively rising PSA levels > 25% above nadir, at least three weeks apart, without evidence of clinical or radiographic progression?",', '        "Q6": "Is the patient\'s PSA doubling time greater than 8 weeks?",', '        "Q7": "Is the patient\'s ongoing systemic treatment deemed feasible by the treating physician?"', '    },', '    "DNF_LOGICAL_EXPRESSION": "(Q1 and Q2 and Q3 and Q4 and Q5 and Q6 and Q7)",', '    "DNF_LOGICAL_EXPRESSION_REASONING": "The patient must meet all inclusion criteria (Q1 to Q7) to 

Generating DNF: 100%|██████████| 17/17 [03:52<00:00, 13.67s/trial]

['```json', '{', '    "QUESTIONS": {', '        "Q1": "Is the patient having a chronic disease?",', '        "Q2": "Is the patient taking formulation wheat grass?",', '        "Q3": "Is the patient taking tulsi?",', '        "Q4": "Is the patient taking wheatgrass and tulsi separately?",', '        "Q5": "Is the patient 30 years or older?",', '        "Q6": "Is the patient using formulations of wheatgrass and tulsi?",', '        "Q7": "Is the patient taking regular medicine?"', '    },', '    "DNF_LOGICAL_EXPRESSION": "((Q1 and Q2 and Q3 and Q4) or (Q2 and Q3 and Q7)) and not Q5 and not Q6",', '    "DNF_LOGICAL_EXPRESSION_REASONING": "A patient is eligible if they have a chronic disease, take wheatgrass, tulsi, and either the formulation or both individually, are 30 or older, use formulations of wheatgrass and tulsi, and are not taking regular medicine.",', '    "INCLUSION_BIOMARKER": "None",', '    "EXCLUSION_BIOMARKER": "None"', '}', '```']


STOP

In [ ]:
# We call the pickle file, then open the patients and per question and patient we do the matching and then we can save the results in a new pickle file.

#load pickle file
with open("path/to/processed_trials.pkl", "rb") as f:
    trials = pickle.load(f)

question = "Q: What are the eligibility criteria for patients with advanced melanoma?"

# collection is the Embeddings of the patient, but we have to open it per patient... 
results = collection.query(
    query_embeddings=[embed_text(question, tokenizer_med, model_med)],
    n_results=20,
    include=["metadatas", "distances"]
)

| Field     | Value                                |
|-----------|---------------------------------------|
| id        | NCT05851235.xml_Q0                    |
| document  | "Is the patient over 18 years old?"   |
| metadata  | {"trial_id": "NCT05851235.xml"}       |
| embedding | MedCPT vector (768-dim)               |

In [34]:
tokenizer = AutoTokenizer.from_pretrained(embedding_model)
tokenizer_model = AutoModel.from_pretrained(embedding_model)
client = embeddings_ClinicalTrials.generate_chromaDB_CT(trials, tokenizer, tokenizer_model, Path(save_dir))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12908.34it/s]


Adding trials to ChromaDB:   0%|          | 0/17 [00:00<?, ?trial/s]

Trial NCT05851547.xml already exists in the collection. Skipping.
Trial NCT05851378.xml already exists in the collection. Skipping.
Trial NCT05851235.xml already exists in the collection. Skipping.
Trial NCT05851014.xml already exists in the collection. Skipping.


Adding trials to ChromaDB:  29%|██▉       | 5/17 [00:05<00:13,  1.15s/trial]

Adding 4 questions to ChromaDB collection.


...

Adding 28 questions to ChromaDB collection.


Adding trials to ChromaDB:  94%|█████████▍| 16/17 [02:18<00:15, 15.00s/trial]

Adding 7 questions to ChromaDB collection.


Adding trials to ChromaDB: 100%|██████████| 17/17 [02:28<00:00,  8.76s/trial]

Adding 7 questions to ChromaDB collection.
